# LearnOS — Tier 2: Deep Knowledge Tracing (DKT)

**Deep learning.** A small LSTM that reads a student's sequence of `(skill, correct/incorrect)` events and predicts mastery across *all* skills at each step.

Why this and not just BKT: BKT treats every skill independently. DKT picks up **cross-topic momentum** — e.g. that struggling with conditional probability predicts struggling with Bayes' theorem. That's the root-cause weakness detection story, backed by a trained model instead of a hand-written rule.

We ship **both**: BKT is the interpretable, cold-start-friendly default; DKT takes over once a student has enough interaction history.

**Output:** `../dkt/dkt_model.pt` (+ `dkt_config.json`)

Trains on CPU in a couple of minutes. Requires `torch`:
```bash
pip install torch
```

In [ ]:
import json
import os
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DKT_DIR = os.path.abspath(os.path.join("..", "dkt"))
os.makedirs(DKT_DIR, exist_ok=True)
print("torch", torch.__version__, "| output dir:", DKT_DIR)

## 1. The skill space

These map to `student_mastery.topic` in Postgres. Keep this list and the DB in sync — the index here *is* the skill id the model uses.

The prerequisite pairs below are what makes the synthetic data worth training on: a student who is weak in `conditional_probability` should also struggle with `naive_bayes`. If the LSTM learns that correlation, it has learned something a per-skill model structurally cannot.

In [ ]:
SKILLS = [
    "probability",              # 0
    "conditional_probability",  # 1
    "naive_bayes",              # 2
    "linear_algebra",           # 3
    "gradient_descent",         # 4
    "recursion",                # 5
    "trees",                    # 6
    "dynamic_programming",      # 7
    "sql_joins",                # 8
    "normalization",            # 9
]
NUM_SKILLS = len(SKILLS)

# (prerequisite, dependent) — mastery of the first lifts the second
PREREQS = [(0, 1), (1, 2), (3, 4), (5, 6), (5, 7), (8, 9)]

for a, b in PREREQS:
    print(f"{SKILLS[a]:26s} -> {SKILLS[b]}")

## 2. Simulate student histories

Completely legitimate for a hackathon POC — and here it's better than legitimate, because we *control* the prerequisite structure, so we can verify the model actually recovered it.

Each simulated student: starts with low mastery per skill, practices a random skill each step, gets it right with probability = current mastery, and learns a little from every attempt. Prerequisite skills leak a fraction of their mastery into their dependents.

In [ ]:
def simulate_student(num_steps=40, seed=None):
    rng = random.Random(seed)
    mastery = [rng.uniform(0.05, 0.35) for _ in range(NUM_SKILLS)]
    # Some students are simply stronger; gives the model varied trajectories
    aptitude = rng.uniform(0.08, 0.22)

    sequence = []
    for _ in range(num_steps):
        skill = rng.randrange(NUM_SKILLS)
        p_correct = min(0.95, mastery[skill])
        correct = rng.random() < p_correct

        sequence.append((skill, int(correct)))

        # Learning happens - more on success than failure
        gain = aptitude if correct else aptitude * 0.4
        mastery[skill] += (1 - mastery[skill]) * gain

        # Prerequisite transfer: strength in a prereq lifts its dependent
        for pre, dep in PREREQS:
            if pre == skill:
                mastery[dep] += (mastery[pre] - mastery[dep]) * 0.06

    return sequence


NUM_STUDENTS = 1200
SEQ_LEN = 40
dataset = [simulate_student(SEQ_LEN, seed=i) for i in range(NUM_STUDENTS)]

print(f"{len(dataset)} students x {SEQ_LEN} steps")
print("\nFirst 8 events of student 0:")
for skill, correct in dataset[0][:8]:
    print(f"  {SKILLS[skill]:26s} {'correct' if correct else 'wrong'}")

## 3. Encode for the model

Standard DKT encoding: the input token is `skill_id * 2 + correctness`, so *"got recursion right"* and *"got recursion wrong"* are distinct tokens. At each step the model predicts the probability of answering **every** skill correctly; we only score it on the skill that was actually attempted next.

In [ ]:
def encode(dataset):
    X, next_skill, next_correct = [], [], []
    for seq in dataset:
        tokens = [s * 2 + c for s, c in seq]
        X.append(tokens[:-1])                       # inputs:  steps 0..n-2
        next_skill.append([s for s, _ in seq[1:]])  # targets: steps 1..n-1
        next_correct.append([c for _, c in seq[1:]])
    return (
        torch.tensor(X, dtype=torch.long),
        torch.tensor(next_skill, dtype=torch.long),
        torch.tensor(next_correct, dtype=torch.float),
    )


X, S, Y = encode(dataset)
split = int(0.85 * len(X))

train_ds = TensorDataset(X[:split], S[:split], Y[:split])
val_ds = TensorDataset(X[split:], S[split:], Y[split:])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

print(f"X {tuple(X.shape)}   train {len(train_ds)}   val {len(val_ds)}")

## 4. The model

In [ ]:
class DKT(nn.Module):
    def __init__(self, num_skills, embed_dim=32, hidden=64):
        super().__init__()
        self.num_skills = num_skills
        self.embed = nn.Embedding(num_skills * 2, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.out = nn.Linear(hidden, num_skills)

    def forward(self, x):
        e = self.embed(x)
        h, _ = self.lstm(e)
        return torch.sigmoid(self.out(self.dropout(h)))  # P(correct) per skill, per step


model = DKT(NUM_SKILLS)
print(model)
print("\nParameters:", sum(p.numel() for p in model.parameters()))

## 5. Train

The loss curve this prints is the artifact to show judges when they ask whether you trained anything.

In [ ]:
def masked_bce(preds, skills, targets):
    """Score only the skill actually attempted at each step."""
    picked = preds.gather(2, skills.unsqueeze(2)).squeeze(2)
    picked = picked.clamp(1e-7, 1 - 1e-7)
    return nn.functional.binary_cross_entropy(picked, targets)


optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
EPOCHS = 30
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, sb, yb in train_dl:
        optimizer.zero_grad()
        loss = masked_bce(model(xb), sb, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_dl)

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, sb, yb in val_dl:
            preds = model(xb)
            val_loss += masked_bce(preds, sb, yb).item()
            picked = preds.gather(2, sb.unsqueeze(2)).squeeze(2)
            correct += ((picked > 0.5).float() == yb).sum().item()
            total += yb.numel()
    val_loss /= len(val_dl)
    val_acc = correct / total

    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_acc": val_acc})
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.1%}")

print(f"\nFinal validation accuracy: {history[-1]['val_acc']:.1%}")

In [ ]:
# Loss curve — screenshot this for the pitch deck
try:
    import matplotlib.pyplot as plt

    epochs = [h["epoch"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
    ax1.plot(epochs, [h["train_loss"] for h in history], label="train")
    ax1.plot(epochs, [h["val_loss"] for h in history], label="val")
    ax1.set_title("DKT training loss"); ax1.set_xlabel("epoch"); ax1.legend()
    ax2.plot(epochs, [h["val_acc"] for h in history], color="teal")
    ax2.set_title("Validation accuracy"); ax2.set_xlabel("epoch")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed - pip install matplotlib to see the curve")
    for h in history[::5]:
        print(h)

## 6. Did it learn the prerequisite structure?

This is the cell that proves the point. Feed the model a student who repeatedly **fails** a prerequisite skill, and check whether its predicted mastery for the *dependent* skill also drops — even though the dependent skill was never practiced.

That drop is the cross-topic signal BKT cannot produce.

In [ ]:
model.eval()

def predict_mastery(events):
    """events: list of (skill_idx, correct). Returns P(correct) for every skill."""
    tokens = torch.tensor([[s * 2 + c for s, c in events]], dtype=torch.long)
    with torch.no_grad():
        return model(tokens)[0, -1].numpy()


PRE, DEP = 1, 2  # conditional_probability -> naive_bayes

failing = [(PRE, 0)] * 10
passing = [(PRE, 1)] * 10

m_fail = predict_mastery(failing)
m_pass = predict_mastery(passing)

print(f"After 10 attempts at '{SKILLS[PRE]}':\n")
print(f"{'skill':28s} {'all wrong':>10s} {'all right':>10s} {'delta':>8s}")
print("-" * 60)
for i, name in enumerate(SKILLS):
    mark = "  <-- dependent" if i == DEP else ("  <-- practiced" if i == PRE else "")
    print(f"{name:28s} {m_fail[i]:10.1%} {m_pass[i]:10.1%} {m_pass[i]-m_fail[i]:+8.1%}{mark}")

print(f"\nTransfer to '{SKILLS[DEP]}': {m_pass[DEP] - m_fail[DEP]:+.1%}")
print("A clear positive delta on the dependent skill = the LSTM recovered the prerequisite link.")

## 7. BKT baseline — the interpretable default

Ship both. BKT needs no training and works from the very first interaction, which is exactly what a new student has. The Progress Engine uses BKT until a student has enough history for DKT to be reliable.

In [ ]:
def bkt_update(p_known, correct, p_learn=0.15, p_slip=0.1, p_guess=0.25):
    """Standard Bayesian Knowledge Tracing posterior + learning step."""
    if correct:
        num = p_known * (1 - p_slip)
        den = num + (1 - p_known) * p_guess
    else:
        num = p_known * p_slip
        den = num + (1 - p_known) * (1 - p_guess)
    posterior = num / den if den > 0 else p_known
    return posterior + (1 - posterior) * p_learn


p = 0.2
print(f"start            {p:.1%}")
for i, correct in enumerate([1, 1, 0, 1, 1, 1], 1):
    p = bkt_update(p, correct)
    print(f"{'correct' if correct else 'wrong':8s} step {i}  {p:.1%}")

## 8. Save the model

In [ ]:
model_path = os.path.join(DKT_DIR, "dkt_model.pt")
config_path = os.path.join(DKT_DIR, "dkt_config.json")

torch.save(model.state_dict(), model_path)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "skills": SKILLS,
        "num_skills": NUM_SKILLS,
        "embed_dim": 32,
        "hidden": 64,
        "prereqs": PREREQS,
        "val_accuracy": history[-1]["val_acc"],
        "final_val_loss": history[-1]["val_loss"],
        "epochs": EPOCHS,
        "train_students": NUM_STUDENTS,
    }, f, indent=2)

print("Saved:")
print(" ", model_path)
print(" ", config_path)
print(f"\nFor the pitch: {history[-1]['val_acc']:.1%} validation accuracy, "
      f"loss {history[0]['val_loss']:.3f} -> {history[-1]['val_loss']:.3f}")

---

## What to say when a judge asks

> **"Is this just prompting an LLM?"**
>
> No. Three deliberate tiers:
> - **Classical ML** — the TF-IDF + Logistic Regression router (notebook 01), with held-out accuracy we can show.
> - **Deep learning** — this DKT-LSTM, with a real loss curve and a demonstrable cross-topic transfer effect BKT structurally cannot capture.
> - **Agentic GenAI** — the specialists themselves, where an LLM is genuinely the right tool and training our own would waste the 24 hours.
>
> We ship BKT *and* DKT: BKT is interpretable and works from interaction one; DKT takes over once there's enough history. The Progress Engine picks whichever is reliable for that student.